<a href="https://colab.research.google.com/github/izaidahmad/Cuda-Practice-Code/blob/main/Copy_of_Untitled2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%%writefile vecadd.c
#include <stdio.h>
#include <stdlib.h>
#include <cuda_runtime.h>



__global__ void vecAddKernel(float *A, float *B, float *C, int N) {
    // Calculate global thread index
    int i = threadIdx.x + blockDim.x * blockIdx.x;


    if (i < N) {
        C[i] = A[i] + B[i];
    }
}

int main() {
    int N = 1024;
    size_t Size = N * sizeof(float);

    // Allocate memory on the Host (CPU)
    float *h_A = (float*)malloc(Size);
    float *h_B = (float*)malloc(Size);
    float *h_C = (float*)malloc(Size);


    for (int i = 0; i < N; i++) {
        h_A[i] = (float)i;
        h_B[i] = (float)(i * 2);
    }


    float *d_A, *d_B, *d_C;
    cudaMalloc((void**)&d_A, Size);
    cudaMalloc((void**)&d_B, Size);
    cudaMalloc((void**)&d_C, Size);


    cudaMemcpy(d_A, h_A, Size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_B, h_B, Size, cudaMemcpyHostToDevice);

    int threadsPerBlock = 256;
    int blocksPerGrid = (N + threadsPerBlock - 1) / threadsPerBlock;


    vecAddKernel<<<blocksPerGrid, threadsPerBlock>>>(d_A, d_B, d_C, N);


    cudaMemcpy(h_C, d_C, Size, cudaMemcpyDeviceToHost);


    printf("Verification of first 5 elements:\n");
    for (int i = 0; i < 5; i++) {
        printf("h_A[%d] + h_B[%d] = %.1f + %.1f = %.1f (Expected C[%d] = %.1f)\n",
               i, i, h_A[i], h_B[i], h_C[i], i, h_A[i] + h_B[i]);
    }


    free(h_A);
    free(h_B);
    free(h_C);
    cudaFree(d_A);
    cudaFree(d_B);
    cudaFree(d_C);

    return 0;
}

Writing vecadd.c


In [ ]:
!nvcc vecadd.cu -o vecadd
!./vecadd

cc1plus: fatal error: vecadd.cu: No such file or directory
compilation terminated.
/bin/bash: line 1: ./vecadd: No such file or directory


In [ ]:
!nvidia-smi
!nvcc --version

Mon Oct 19 04:38:07 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   46C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
%%writefile vecadd.cu

#include <stdio.h>
#include <cuda_runtime.h>

__global__ void vecAddKernel(float *A, float *B, float *C, int N)
{
    // Calculate global thread index
    int i = threadIdx.x + blockDim.x * blockIdx.x;

    if (i < N)
    {
        C[i] = A[i] + B[i];
    }
}

int main()
{
    int N = 5;
    size_t bytes = N * sizeof(float);

    float h_A[5] = {1, 2, 3, 4, 5};
    float h_B[5] = {10, 20, 30, 40, 50};
    float h_C[5];

    float *d_A, *d_B, *d_C;

    // Allocate GPU memory
    cudaMalloc((void**)&d_A, bytes);
    cudaMalloc((void**)&d_B, bytes);
    cudaMalloc((void**)&d_C, bytes);

    // Copy data from CPU to GPU
    cudaMemcpy(d_A, h_A, bytes, cudaMemcpyHostToDevice);
    cudaMemcpy(d_B, h_B, bytes, cudaMemcpyHostToDevice);

    // Launch CUDA kernel
    vecAddKernel<<<1, N>>>(d_A, d_B, d_C, N);

    // Wait for GPU computation to finish
    cudaDeviceSynchronize();

    // Copy result from GPU to CPU
    cudaMemcpy(h_C, d_C, bytes, cudaMemcpyDeviceToHost);

    // Print results
    for (int i = 0; i < N; i++)
    {
        printf("%.0f ", h_C[i]);
    }

    printf("\n");

    // Free GPU memory
    cudaFree(d_A);
    cudaFree(d_B);
    cudaFree(d_C);

    return 0;
}

In [ ]:
!nvcc vecadd.cu -o vecadd

cc1plus: fatal error: vecadd.cu: No such file or directory
compilation terminated.


In [ ]:
!./vecadd

/bin/bash: line 1: ./vecadd: No such file or directory


In [ ]:
!1s

/bin/bash: line 1: 1s: command not found


In [ ]:
!ls


sample_data  vecadd.c


In [ ]:
!mv vecadd.c vecadd.cu

In [ ]:
!nvcc vecadd.cu -o vecadd

In [ ]:
!./vecadd

Verification of first 5 elements:
h_A[0] + h_B[0] = 0.0 + 0.0 = 0.0 (Expected C[0] = 0.0)
h_A[1] + h_B[1] = 1.0 + 2.0 = 3.0 (Expected C[1] = 3.0)
h_A[2] + h_B[2] = 2.0 + 4.0 = 6.0 (Expected C[2] = 6.0)
h_A[3] + h_B[3] = 3.0 + 6.0 = 9.0 (Expected C[3] = 9.0)
h_A[4] + h_B[4] = 4.0 + 8.0 = 12.0 (Expected C[4] = 12.0)


In [ ]:
%%writefile vecadd.cu

#include <stdio.h>
#include <cuda_runtime.h>

__global__ void vecAddKernel(float *A, float *B, float *C, int N)
{
    // Calculate global thread index
    int i = threadIdx.x + blockDim.x * blockIdx.x;

    if (i < N)
    {
        C[i] = A[i] - B[i];
    }
}

int main()
{
    int N = 5;
    size_t bytes = N * sizeof(float);

    float h_A[5] = {4, 7, 5, 8, 7};
    float h_B[5] = {10, 20, 30, 40, 50};
    float h_C[5];

    float *d_A, *d_B, *d_C;

    // Allocate GPU memory
    cudaMalloc((void**)&d_A, bytes);
    cudaMalloc((void**)&d_B, bytes);
    cudaMalloc((void**)&d_C, bytes);

    // Copy data from CPU to GPU
    cudaMemcpy(d_A, h_A, bytes, cudaMemcpyHostToDevice);
    cudaMemcpy(d_B, h_B, bytes, cudaMemcpyHostToDevice);

    // Launch CUDA kernel
    vecAddKernel<<<1, N>>>(d_A, d_B, d_C, N);

    // Wait for GPU computation to finish
    cudaDeviceSynchronize();

    // Copy result from GPU to CPU
    cudaMemcpy(h_C, d_C, bytes, cudaMemcpyDeviceToHost);

    // Print results
    for (int i = 0; i < N; i++)
    {
        printf("%.0f ", h_C[i]);
    }

    printf("\n");

    // Free GPU memory
    cudaFree(d_A);
    cudaFree(d_B);
    cudaFree(d_C);

    return 0;
}

Overwriting vecadd.cu


In [ ]:
!nvcc vecadd.cu -o vecadd

In [ ]:
!./vecadd

-6 -13 -25 -32 -43 


In [ ]:
%%writefile vecMultiKernel
#include <stdio.h>
#include <cuda_runtime.h>

__global__ vecMultiKernel(float* A, float B, float C, n){
  float i = blockIdx.x * blockDim.x + threadIdx.x;
  if(i<n){
    C[i] * A[i] * B[i];
  }
}
int main(){
  int n = 5;
    size_t bytes = N * sizeof(float);

    float host_A[5] = {12, 14, 20, 15, 20};
    float host_B[5] = {12, 10, 5, 8, 9};
    float host_C[5];
    cudaMalloc((void**)&device_A, bytes);
    cudaMalloc((void**)&device_B, bytes);

    cudaMemcpy(device_A,host_A,bytes, cudaMemcpyHostToDevice);
    cudaMemcpy(device_B,host_B,bytes, cudaMemcpyHostToDevice);
    cudaMemcpy(host_C,device_C,bytes,cudaMemcpyDeviceToHost);
    int threadsPerBlock = 256;
    int bock = (n+threadsPerBlock-1)/threadsPerBlock;

    vecMultiKernel<<<1,256.0>>>(device_A,device_B,device_C, n);
     cudaMalloc((void**)&device_C, bytes);
     printf("Result");
     for(int i = 0; i<n; i++){
      printf("%d",host_C[i]);
      printf("\n");
     }

      cudaFree(device_A);
       cudaFree(device_B);
        cudaFree(device_C);
        free(host_A);
        free(host_B);
        free(host_C);



return 0;


}


Writing vecMultiKernel


In [ ]:
!nvcc vecadd.cu -o vecMultiKernel

In [ ]:
!./vecMultiKernel

-6 -13 -25 -32 -43 
